In [1]:
#Customer Value, Loyalty and Shopping Preference Analysis using Python, SQL Server and Power BI

In [2]:
#Problem Statement: 
#A retail company wants to understand its customers’ shopping behaviour, product preferences and 
#purchasing patterns. This project analyzes customer demographics, purchase amounts, subscriptions, 
#discounts, product categories and shopping preferences to identify valuable customer segments, 
#high-performing products and opportunities for improving marketing and subscription strategies."

In [3]:
#importing necessary libraries:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None) #To display all cols in a pandas df without truncation

In [4]:
#Data Loading:
# Load the customer shopping dataset

file_path = "Desktop/Customer_Shopping_Project/data/customer_shopping_behavior (1).csv"

df = pd.read_csv(file_path)

df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [46]:
## 1. Data Understanding and Profiling:

In [5]:
df.shape

(3900, 18)

In [6]:
df.info() # Display dataset structure and data types

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [7]:
df.describe(include='all') #df.describe() gives the statistical summarry of numerical cols. If we also want for categorical cols we write include='all'

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [8]:
#Now, we'll check for missing values
df.isnull().sum() #We find that review rating has 37 null values

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [9]:
#Now we'll replace the missing values. We'll choose median over mean because median is robust to outliers
#But review can vary a lot depending on the category. So, we'll impute the missing values using median review rating within each category.

# For this first we preserve the original dataset
clean_df = df.copy()

# Then we first find the corresponding category median for every row
category_medians = (
    clean_df.groupby("Category")["Review Rating"] .transform("median")
    #groupby("Category") logically divides the rows into groups:Clothing,Accessories,Footwear,Outerwear
    #clean_df.groupby("Category")["Review Rating"]: Within each category, Pandas now considers only the Review Rating values.
    #.transform("median") :Pandas calculates the median rating separately for each category.
)

# Then we replace missing ratings with those category medians
clean_df["Review Rating"] = (
    clean_df["Review Rating"].fillna(category_medians)
)

In [10]:
clean_df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [11]:
#Now, we'll check duplicates
clean_df.duplicated().sum() #We find that there are zero duplicates

np.int64(0)

In [12]:
#Now, we'll standardize column names
clean_df.columns = (
    clean_df.columns
    .str.strip() #removes unnecessary leading and trailing spaces but does not remove spaces between words
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[()]", "", regex=True) #This removes both opening and closing parentheses. regex=True tells Pandas to interpret [()] as a regular-expression pattern.
)

In [13]:
clean_df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_usd', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [14]:
#Now, we'll do feature engineering and create a column age group because the original Age column contains 53 distinct values, 
#which will make customer comparisons and dashboard visuals too granular. Individual age was useful for distributions and correlations, 
#while age groups are more suitable for customer segmentation and business reporting
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
clean_df['age_group'] = pd.qcut(clean_df['age'], q=4, labels=labels) #This qcut splits the age group into 4 equal sized parts and assigns the labels we've defined

In [15]:
clean_df[['age', 'age_group']].head()

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged


In [16]:
# We'll check the age range and customer count in each group

age_group_summary = (
    clean_df.groupby(
        "age_group",
        observed=False
    )["age"]
    .agg(["count", "min", "max"])
)

age_group_summary

,count,min,max
age_group,,,
Young Adult,1028,18,31
Adult,942,32,44
Middle-aged,986,45,57
Senior,944,58,70


In [18]:
#Now, we'll use feature-engineering to classify recorded purchases into low-, medium- and high-value groups:
#We Calculate the 25th and 75th percentile purchase amounts
lower_spend_threshold = (
    clean_df["purchase_amount_usd"].quantile(0.25)
)

upper_spend_threshold = (
    clean_df["purchase_amount_usd"].quantile(0.75)
)

print("Lower spending threshold:", lower_spend_threshold)
print("Upper spending threshold:", upper_spend_threshold)

Lower spending threshold: 39.0
Upper spending threshold: 81.0


In [20]:
# Now we'll create spending segments using the calculated thresholds:
spending_conditions = [
    clean_df["purchase_amount_usd"] <= lower_spend_threshold,
    clean_df["purchase_amount_usd"] >= upper_spend_threshold
]

spending_labels = [
    "Low-Value Purchase",
    "High-Value Purchase"
]

clean_df["spending_segment"] = np.select(
    spending_conditions,
    spending_labels,
    default="Medium-Value Purchase"
)

In [21]:
clean_df[["purchase_amount_usd", "spending_segment"]].head(10)

,purchase_amount_usd,spending_segment
0,53,Medium-Value Purchase
1,64,Medium-Value Purchase
2,73,Medium-Value Purchase
3,90,High-Value Purchase
4,49,Medium-Value Purchase
5,20,Low-Value Purchase
6,85,High-Value Purchase
7,34,Low-Value Purchase
8,97,High-Value Purchase
9,31,Low-Value Purchase


In [22]:
spending_segment_summary = (
    clean_df.groupby("spending_segment")[
        "purchase_amount_usd"
    ]
    .agg(["count", "min", "max", "mean"])
    .round(2)
)

spending_segment_summary
#This helps us to calculate high-value customer count, revenue contribution, subscription rate and discount usage by spending segment.

,count,min,max,mean
spending_segment,,,,
High-Value Purchase,978,81,100,90.55
Low-Value Purchase,1014,20,39,29.52
Medium-Value Purchase,1908,40,80,60.06


In [23]:
#Right now the dataset has
frequency_mapping = {
    'Weekly': 7,
    'Fortnightly': 14,
    'Monthly': 30,
    'Quarterly': 90,
    'Annually': 365,
    'Every 3 months': 90
}
clean_df['purchase_frequency_days'] = clean_df['frequency_of_purchases'].map(frequency_mapping) 

In [24]:
clean_df['purchase_frequency_days'].head(10)

0     14.0
1     14.0
2      7.0
3      7.0
4    365.0
5      7.0
6     90.0
7      7.0
8    365.0
9     90.0
Name: purchase_frequency_days, dtype: float64

In [25]:
clean_df[['item_purchased','discount_applied']].head(10)

,item_purchased,discount_applied
0,Blouse,Yes
1,Sweater,Yes
2,Jeans,Yes
3,Sandals,Yes
4,Blouse,Yes
5,Sneakers,Yes
6,Shirt,Yes
7,Shorts,Yes
8,Coat,Yes
9,Handbag,Yes


In [26]:
#Now we'll export our cleaned clean_df from Jupyter as a CSV so that we can import it into SQL Server.
clean_df.to_csv("customer_shopping_cleaned.csv", index=False)
#You want to export your cleaned clean_df from Jupyter as a CSV so you can import it into SQL Server.

In [37]:
#import os
#print(os.getcwd())

In [2]:
%pip install sqlalchemy pyodbc  #Pandas/Jupyter cannot use SQL Server through SQLAlchemy unless the required libraries are available.

Note: you may need to restart the kernel to use updated packages.


In [30]:
## Load Cleaned Data into SQL Server

##After completing data cleaning and feature engineering in Python,the cleaned dataset was loaded into SQL Server for SQL-based analysis.
from sqlalchemy import create_engine
from urllib.parse import quote_plus #quote_plus is needed because our ODBC connection string contains special characters that need to be safely passed inside the SQLAlchemy URL

connection_string = quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=CustomerShoppingDB;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_string}"
)

In [31]:
clean_df.to_sql(
    "customer_shopping",
    con=engine,
    if_exists="replace",
    index=False
)

print("Cleaned data successfully loaded into SQL Server.")

E:\Users\USER\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Cleaned data successfully loaded into SQL Server.
